In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import scanpy as sc
import os
import pandas as pd
import seaborn as sns
import numpy as np
import ktplotspy as kpy

In [ ]:
from pyfuncs.general import set_plotting_style
from pyfuncs.general import magma

set_plotting_style()

In [ ]:
os.makedirs('results/ARAUZO_03/cellphonedb/files', exist_ok=True)

---

In [ ]:
adata = sc.read_h5ad('data/ARAUZO_03/processed_adatas/adata_processed.h5ad')
adata_FAP = sc.read_h5ad('data/ARAUZO_03/processed_adatas/adata_FAP.h5ad')

In [ ]:
# Modify adatas to include human orthologs 
biomart_df = pd.read_csv('database/biomart/mouse_human_orthologues.txt', sep='\t')
biomart_df.fillna('', inplace=True)

mouse_human_orthologue_dict = dict(zip(biomart_df['Gene name'], biomart_df['Human gene name']))


In [ ]:
human_var_names_adata = adata.var_names.map(lambda x: mouse_human_orthologue_dict.get(x, ''))
map_idx = human_var_names_adata != ''
adata = adata[:, map_idx]
adata.var_names = human_var_names_adata[map_idx]

In [ ]:
human_var_names_adata_FAP = adata_FAP.var_names.map(lambda x: mouse_human_orthologue_dict.get(x, ''))
map_idx = human_var_names_adata_FAP != ''
adata_FAP = adata_FAP[:, map_idx]
adata_FAP.var_names = human_var_names_adata_FAP[map_idx]

In [ ]:
adata.write('data/ARAUZO_03/processed_adatas/adata_processed_human_orthologs.h5ad')
adata_FAP.write('data/ARAUZO_03/processed_adatas/adata_FAP_processed_human_orthologs.h5ad')

## Configure CellPhoneDB 

In [ ]:
from IPython.display import HTML, display
from cellphonedb.utils import db_releases_utils, db_utils

display(HTML(db_releases_utils.get_remote_database_versions_html()['db_releases_html_table']))

In [ ]:
# -- Version of the databse
cpdb_version = 'v5.0.0'

# -- Path where the input files to generate the database are located
cpdb_target_dir = os.path.join('results/ARAUZO_03/cellphonedb/db', cpdb_version)
db_utils.download_database(cpdb_target_dir, cpdb_version)

# Example of CellPhoneDB v5 run

In [ ]:
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

In [ ]:
cpdb_files_dir = 'results/ARAUZO_03/cellphonedb/files'

In [ ]:
df_meta = adata_FAP.obs['cats_3tier'].reset_index()
df_meta.rename(columns={'cats_3tier': 'cell_type', 'barcode': 'barcode_sample'}, inplace=True)
df_meta.to_csv(f'{cpdb_files_dir}/test_meta.txt', sep='\t', index=False)
df_meta

In [ ]:
# Method 2

cpdb_results = cpdb_statistical_analysis_method.call(
    cpdb_file_path = f"{cpdb_target_dir}/cellphonedb.zip",                 # mandatory: CellphoneDB database zip file.
    meta_file_path = f'{cpdb_files_dir}/test_meta.txt',                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = 'data/ARAUZO_03/processed_adatas/adata_FAP_processed_human_orthologs.h5ad',             # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                       # optional: whether to score interactions or not. 
    iterations = 1000,                               # denotes the number of shufflings performed in the analysis.
    threshold = 0.4,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 5,                                     # number of threads to use in the analysis.
    debug_seed = 42,                                 # debug randome seed. To disable >=0.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    pvalue = 0.001,                                   # P-value threshold to employ for significance.
    subsampling = False,                             # To enable subsampling the data (geometri sketching).
    subsampling_log = False,                         # (mandatory) enable subsampling log1p for non log-transformed data inputs.
    subsampling_num_pc = 100,                        # Number of componets to subsample via geometric skectching (dafault: 100).
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = f'{cpdb_files_dir}/results_test_method2',                          # Path to save results.
    output_suffix = None                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

In [ ]:
kpy.plot_cpdb_heatmap(pvals = cpdb_results['pvalues'],
                      degs_analysis = False,
                      figsize = (5, 5),
                      title = "Sum of significant interactions", 
                      cmap='Blues', 
                      annot=True)

In [ ]:
cpdb_results['significant_means'].sort_index()

In [ ]:
df_means = cpdb_results["means"].copy()

mask_pval = cpdb_results["pvalues"].iloc[:, 13:] > 0.001
df_means.iloc[:, 13:] = df_means.iloc[:, 13:].mask(mask_pval, np.nan)

In [ ]:
sns.histplot(df_means.iloc[:, 13:].values.flatten(), bins=100)

In [ ]:
mask_mean = cpdb_results["means"].iloc[:, 13:] < 1
df_means.iloc[:, 13:] = df_means.iloc[:, 13:].mask(mask_mean, np.nan)

In [ ]:
df_means_filter = df_means.loc[df_means.iloc[:, 13:].notna().any(axis=1)]
df_means_filter

In [ ]:
from cellphonedb.utils import search_utils

search_results = search_utils.search_analysis_results(
    query_cell_types_1 = ['FAP_A'],  # List of cells 1, will be paired to cells 2 (list or 'All').
    query_cell_types_2 = ['FAP_B'],     # List of cells 2, will be paired to cells 1 (list or 'All').
    query_genes = ['IGF1'],                                       # filter interactions based on the genes participating (list).
    query_interactions = ['IGF1_IGFR1'],                            # filter intereactions based on their name (list).
    significant_means = cpdb_results['significant_means'],          # significant_means file generated by CellphoneDB.
    deconvoluted = cpdb_results['deconvoluted'],                    # devonvoluted file generated by CellphoneDB.
    interaction_scores = cpdb_results['interaction_scores'],        # interaction score generated by CellphoneDB.
    query_minimum_score = 50,                                       # minimum score that an interaction must have to be filtered.
    separator = '|',                                                # separator (default: |) employed to split cells (cellA|cellB).
    long_format = True,                                             # converts the output into a wide table, removing non-significant interactions
    query_classifications = ['Signaling by Transforming growth factor']
)

search_results.head()

In [ ]:
kpy.plot_cpdb(
    adata = adata_FAP,
    cell_type1 = "FAP_A|FAP_B",
    cell_type2 = "FAP_CDEF",
    means = cpdb_results['means'],
    pvals = cpdb_results['pvalues'],
    celltype_key = "cats_3tier",
    genes = ["IGF1", "IGF1R"],
    figsize = (10, 3),
    title = "Interactions",
    max_size = 3,
    highlight_size = 0.75,
    degs_analysis = False,
    standard_scale = True,
    interaction_scores = cpdb_results['interaction_scores'],
    scale_alpha_by_interaction_scores = True
)